**Step 5: Experiment 1, Compare the Candidate Generators**

This notebook runs Experiment 1 of the experiment design: it compares the eight candidate generators, the methods that decide which archive names resemble a query, each on its own with no dictionary. Every generator answers every validation query against the whole archive (all persons) and is scored by **R@5**, the share of a query's correct answers among the first five persons returned, averaged per query set and then over the five sets. Each embedding model's cut-off is tuned on the same queries. It reads the outputs of steps 1–4 and outputs:

1. `./outputs/step_5_generator_results.csv`: one row per generator with its tuned cut-off, its R@5 per query set and macro-averaged, its mean reciprocal rank (MRR), and whether it is carried into Experiment 2 (the two best embedding models and the best lexical method)

**1. Configuration and Data**

This cell, the same in steps 5–8, reads what every search needs and builds the archive index the queries are answered from (all persons):

1. `forms`: the vocabulary, the archive's spellings first (`N_ARCHIVE` of them), then the dictionary spellings found in no record, each with its display spelling
2. the token arrays: every name token of every person, with its person, field (given name or surname), spelling and column, sorted so that each person's tokens of one field sit together
3. `dictionaries`: for D1, D2 and D3 of step 3, each group's spellings and each spelling's groups
4. the signals of step 4: every model's embedding of every spelling, the archive spellings having each phonetic code, and the trigram similarities
5. `load_queries`: one split's queries of step 2, each with its tokens, its correct answers and the index tokens hidden from it (for the cross-script sets QS3 and QS4, the target's tokens in the query's own script)

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and data  —  the archive index, the dictionaries, the signals
#
# This cell is the same in steps 5–8. A spelling is identified by its
# position in `forms`. The archive's spellings come first, so an array over
# the first N_ARCHIVE positions covers every spelling a person has.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import functools
import json
import time
from collections import defaultdict
from pathlib import Path

import numpy as np

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
OUTPUTS_DIR  = Path("outputs")                    # every step's outputs
QUERY_SETS   = ("QS1", "QS2", "QS3", "QS4", "QS5")
EMBEDDINGS   = ("e1", "e2", "e3a", "e3b")         # the embedding models
LEXICAL      = ("l1", "l2", "l3", "l4")           # the lexical methods
DICTIONARIES = ("d1", "d2", "d3")
MATCH_CAP    = 50        # embedding matches kept per query token
P3_TOP_K     = 5         # neighbours of P3's fixed-size retrieval
RRF_K        = 60        # the constant of reciprocal rank fusion (K3)
TOP_K        = 50        # persons ranked per query
RULE         = "─" * 72

# A component of a search configuration, with nothing set (see the next cell).
COMPONENT = {"generator": "", "cutoff": None, "placement": "", "dictionary": "",
             "vocab": "", "retrieval": "", "tau_link": None}


def read_csv(path: Path) -> list[dict]:
    """Read the CSV file `path` into a list of dictionaries."""
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def write_csv(path: Path, rows: list[dict]) -> None:
    """Write `rows` (dictionaries with the same keys) to the CSV file `path`."""
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


# ── The persons and the archive's spellings ──────────────────────────────────
persons = sorted(read_csv(OUTPUTS_DIR / "step_1_persons.csv"), key=lambda p: p["person_id"])
person_index = {p["person_id"]: i for i, p in enumerate(persons)}
codes = {(r["script"], r["norm"]): r for r in read_csv(OUTPUTS_DIR / "step_4_name_codes.csv")}
forms = [(s, n, r["display"]) for (s, n), r in codes.items() if r["in_archive"] == "1"]
N_ARCHIVE = len(forms)
form_index = {(s, n): i for i, (s, n, _) in enumerate(forms)}

# ── The dictionaries, whose new spellings extend the vocabulary ──────────────
dictionaries = {}
for _name in DICTIONARIES:
    _groups = defaultdict(set)
    for _row in read_csv(OUTPUTS_DIR / f"step_3_dictionary_{_name}.csv"):
        _key = (_row["script"], _row["norm"])
        if _key not in form_index:
            form_index[_key] = len(forms)
            forms.append((*_key, _row["display"]))
        _groups[_row["group_id"]].add(form_index[_key])
    _groups_of = defaultdict(set)
    for _group, _members in _groups.items():
        for _form in _members:
            _groups_of[_form].add(_group)
    dictionaries[_name] = {"groups": dict(_groups), "groups_of": dict(_groups_of)}

# ── The archive index: one entry per name token of every person ──────────────
_tokens = sorted((t["field"] == "surname", person_index[p["person_id"]],
                  form_index[(t["script"], t["norm"])], t["column"])
                 for p in persons for t in json.loads(p["tokens"]))
tok_surname, tok_person = np.array([t[0] for t in _tokens]), np.array([t[1] for t in _tokens])
tok_form, tok_column = np.array([t[2] for t in _tokens]), np.array([t[3] for t in _tokens])
# where each run of one person's given-name (or surname) tokens starts
seg_start = np.flatnonzero(np.r_[True, (np.diff(tok_person) != 0) | (np.diff(tok_surname) != 0)])
form_token_count = np.bincount(tok_form, minlength=N_ARCHIVE)


# ── The signals of step 4 ────────────────────────────────────────────────────
def load_embeddings(name: str) -> tuple[dict, np.ndarray]:
    """Return a model's text → vector map and the vectors of the whole vocabulary."""
    stored = np.load(OUTPUTS_DIR / f"step_4_embeddings_{name}.npz")
    vector_of = dict(zip(map(str, stored["texts"]), stored["vectors"]))
    return vector_of, np.stack([vector_of[display] for _, _, display in forms])


def code_index(column: str) -> dict[str, list[int]]:
    """Return an inverted index from a phonetic code to the archive spellings having it."""
    index = defaultdict(list)
    for i, (script, norm, _) in enumerate(forms[:N_ARCHIVE]):
        for code in codes[(script, norm)][column].split():
            index[code].append(i)
    return index


EMBEDDING_VECTORS = {name: load_embeddings(name) for name in EMBEDDINGS}
CODE_INDEXES = {column: code_index(column) for column in ("dm_codes", "bmpm_approx",
                                                          "bmpm_hebrew")}
_trigram = np.load(OUTPUTS_DIR / "step_4_trigram_similarities.npz")
TRIGRAMS = defaultdict(list)       # query token "script|norm" → [(archive spelling, value)]
for _row, _col, _value in zip(_trigram["rows"], _trigram["cols"], _trigram["values"]):
    TRIGRAMS[str(_trigram["query_keys"][_row])].append((form_index[tuple(
        str(_trigram["archive_keys"][_col]).split("|"))], float(_value)))


def load_queries(split: str) -> list[dict]:
    """Return the queries of one split ("validation" or "test").

    Each query holds its id, set, tokens, target person, correct answers
    (`relevant`, person positions) and `masked`, the index tokens hidden from it.
    """
    loaded = []
    for row in read_csv(OUTPUTS_DIR / "step_2_queries.csv"):
        if row["split"] == split:
            target = person_index[row["target_person_id"]]
            loaded.append({"id": row["query_id"], "set": row["set"], "text": row["query_text"],
                            "tokens": json.loads(row["tokens"]), "target": target,
                            "relevant": {person_index[p]
                                         for p in json.loads(row["relevant_person_ids"])},
                            "masked": np.flatnonzero((tok_person == target) & np.isin(
                                tok_column, row["masked_columns"].split(";")))})
    return loaded


print(f"✓ Configuration and data loaded ({len(persons)} persons, {len(tok_form)} name tokens, "
      f"{N_ARCHIVE} archive and {len(forms) - N_ARCHIVE} dictionary-only spellings).")

✓ Configuration and data loaded (637 persons, 2309 name tokens, 1206 archive and 421 dictionary-only spellings).


**2. The Search Engine**

This cell, the same in steps 5–8, defines how a search configuration answers a query and how its answers are scored. A configuration has one or more **components**, a **ranker** and an **aggregation**:

1. **Generators**: a component's generator gives the similarity of a query token to every archive spelling. The embedding models (E1, E2, E3a, E3b) give the cosine of the two embeddings, keeping the matches with a cosine of at least the cut-off `c`, at most 50 per query token. The lexical methods give 1 for a match: L1 for the same normalised spelling, L3 for a shared Daitch–Mokotoff code, L4 for a shared Beider–Morse token. L2 gives the trigram similarity, within one script
2. **Dictionary placements**: a component may use a dictionary, so that its matches are either direct (found by the generator) or dictionary-derived. Every archive spelling is linked to the dictionary groups of its own spelling or, for P3, to those of the most similar dictionary spelling if their cosine reaches `tau_link`. Then:
   - **P1, dictionary only**: the token's own spelling, plus every archive spelling linked to the token's groups
   - **P2, dictionary first**: the token and the other spellings of its groups are each matched by the embedding model; an archive spelling's derived similarity is its best cosine to one of those spellings
   - **P3, embedding first**: the token's nearest spellings by embedding (cosine of at least `c`, or the five nearest), searched among the archive's spellings or among those and the dictionary's, are direct matches, and bring the archive spellings linked to their groups at their cosine
3. **Rankers**: K1 gives each index token its best similarity over the components; K2 does the same with dictionary-derived matches weighted by λ, so a direct match of equal strength ranks first; K3, reciprocal rank fusion, lets each component rank the persons on its own and scores a person Σ 1 / (60 + rank)
4. **Person scoring**: for each query token, a person's given-name score is its best similarity among its given-name tokens, and likewise for the surname. A one-token query scores `max(g, s)`; a longer query scores the best `agg(g_i, s_j)` over pairs of different query tokens, with `agg` the sum or the product, so that a full name ranks a person highly only when both parts match
5. **Metrics**: **R@5**, the share of a query's correct answers among the first five persons returned (out of at most five), and the reciprocal rank of the first correct answer among the first 50 (0 if none), each averaged per query set and then over the five sets (the macro-average; the reciprocal rank's is the MRR)

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Search engine  —  components, rankers, person scores and metrics
#
# This cell is the same in steps 5–8. A configuration is a dictionary:
#   components   a list of COMPONENT dictionaries: a `generator` ("e1",
#                "l3", ...; "" for P1), its `cutoff`, and a `placement`
#                ("" for none, "P1", "P2", "P3") with its `dictionary`,
#                and P3's `vocab`, `retrieval` and `tau_link`
#   ranker       "K1", "K2" or "K3"
#   lambda       the weight of a dictionary-derived match (K2, and K3
#                inside each component)
#   aggregation  "sum" or "product"
# A spelling held only by a query's hidden tokens is left out of the
# vocabulary for that query, so a cross-script query cannot reach its
# target through the spelling it is meant not to know.
# ─────────────────────────────────────────────────────────────────────────────
if "load_queries" not in globals():
    raise RuntimeError("Run the 'Configuration and Data' cell first.")

_LINKS = {}      # links per (generator, dictionary, tau_link), built once


@functools.cache
def cosines(generator: str, text: str) -> np.ndarray:
    """Return the cosine of `text` to every vocabulary spelling (negatives as 0).

    The result is cached, so callers must not change it in place.
    """
    vector_of, vocabulary = EMBEDDING_VECTORS[generator]
    return np.clip(vocabulary @ vector_of[text], 0, None)


def keep_matches(similarity: np.ndarray, cutoff: float) -> np.ndarray:
    """Zero the similarities under `cutoff`, then all but the MATCH_CAP largest."""
    similarity = np.where(similarity >= cutoff, similarity, 0)
    similarity[np.argsort(-similarity, kind="stable")[MATCH_CAP:]] = 0
    return similarity


def generator_similarity(generator: str, token: dict, cutoff: float | None,
                         available: np.ndarray) -> np.ndarray:
    """Return a generator's similarity of one query token to every archive spelling.

    Args:
        generator: an embedding model or a lexical method.
        token: the query token, with its script, norm and display spelling.
        cutoff: the lowest cosine an embedding match may have.
        available: True for each archive spelling the query may match.
    """
    similarity, key = np.zeros(N_ARCHIVE), (token["script"], token["norm"])
    if generator in EMBEDDINGS:
        similarity = keep_matches(cosines(generator, token["display"])[:N_ARCHIVE] * available,
                                  cutoff)
    elif generator == "l1" and form_index.get(key, N_ARCHIVE) < N_ARCHIVE:
        similarity[form_index[key]] = 1.0
    elif generator == "l2":
        for form, value in TRIGRAMS[f"{key[0]}|{key[1]}"]:
            similarity[form] = value
    elif generator in ("l3", "l4"):
        for column in (("dm_codes",) if generator == "l3" else ("bmpm_approx", "bmpm_hebrew")):
            for code in codes[key][column].split():
                similarity[CODE_INDEXES[column].get(code, [])] = 1.0
    return similarity * available


def links(generator: str, dictionary: str, tau_link: float | None) -> tuple[dict, list]:
    """Link every archive spelling to dictionary groups: those of its own spelling
    or, with a `tau_link`, those of the dictionary spelling most similar to it
    by embedding, if their cosine reaches `tau_link`.

    Returns:
        Each group's linked archive spellings, and each archive spelling's groups.
    """
    if (generator, dictionary, tau_link) not in _LINKS:
        groups_of = dictionaries[dictionary]["groups_of"]
        dictionary_forms = np.array(sorted(groups_of))
        # without a tau_link (P1), no embedding is needed: every cosine counts as 0
        vectors = EMBEDDING_VECTORS[generator][1] if tau_link else np.zeros((len(forms), 1))
        similarity = vectors[:N_ARCHIVE] @ vectors[dictionary_forms].T
        members, groups_of_form = defaultdict(list), []
        for form in range(N_ARCHIVE):
            groups = groups_of.get(form, set())
            if not groups and tau_link and similarity[form].max() >= tau_link:
                groups = groups_of[dictionary_forms[np.argmax(similarity[form])]]
            groups_of_form.append(groups)
            for group in groups:
                members[group].append(form)
        _LINKS[(generator, dictionary, tau_link)] = (dict(members), groups_of_form)
    return _LINKS[(generator, dictionary, tau_link)]


def p3_neighbours(component: dict, token: dict,
                  available: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return P3's nearest spellings to one query token by embedding, and their
    cosines: those with a cosine of at least the cut-off, or the five nearest."""
    vocabulary = np.flatnonzero(available)
    if component["vocab"] == "archive+dict":
        vocabulary = np.union1d(vocabulary,
                                sorted(dictionaries[component["dictionary"]]["groups_of"]))
    similarity = cosines(component["generator"], token["display"])[vocabulary]
    order = np.argsort(-similarity, kind="stable")
    if component["retrieval"] == "top5":
        nearest = order[:P3_TOP_K]
    else:
        nearest = order[similarity[order] >= component["cutoff"]][:MATCH_CAP]
    return vocabulary[nearest], similarity[nearest]


def p3_matches(component: dict, token: dict, available: np.ndarray,
               own_groups: set) -> tuple[np.ndarray, np.ndarray]:
    """Return P3's direct and derived similarity of one query token to every
    archive spelling: its nearest spellings are direct matches, and they (and
    the token's own groups, at 1) bring the spellings linked to their groups."""
    groups_of = dictionaries[component["dictionary"]]["groups_of"]
    members, groups_of_form = links(component["generator"], component["dictionary"],
                                    component["tau_link"])
    direct, derived, seeds = np.zeros(N_ARCHIVE), np.zeros(N_ARCHIVE), [(1.0, own_groups)]
    for form, cosine in zip(*p3_neighbours(component, token, available)):
        if form < N_ARCHIVE:
            direct[form] = cosine
            seeds.append((cosine, groups_of_form[form]))
        else:
            seeds.append((cosine, groups_of[form]))
    for cosine, groups in seeds:
        for group in groups:
            linked = members.get(group, [])
            derived[linked] = np.maximum(derived[linked], cosine)
    return direct, derived


def component_matches(component: dict, token: dict,
                      available: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return a component's direct and dictionary-derived similarity of one query
    token to every archive spelling."""
    if not component["placement"]:
        return (generator_similarity(component["generator"], token, component["cutoff"],
                                     available), np.zeros(N_ARCHIVE))
    dictionary = dictionaries[component["dictionary"]]
    own = form_index.get((token["script"], token["norm"]))
    own_groups = dictionary["groups_of"].get(own, set())
    if component["placement"] == "P3":
        return p3_matches(component, token, available, own_groups)
    if component["placement"] == "P1":
        members, _ = links("", component["dictionary"], None)
        derived = np.zeros(N_ARCHIVE)
        for group in own_groups:
            derived[members.get(group, [])] = 1.0
        return generator_similarity("l1", token, None, available), derived * available
    mates = set().union(*(dictionary["groups"][g] for g in own_groups)) - {own}
    best = np.max([cosines(component["generator"], forms[m][2])[:N_ARCHIVE] for m in mates],
                  axis=0) if mates else np.zeros(N_ARCHIVE)
    return (generator_similarity(component["generator"], token, component["cutoff"], available),
            keep_matches(best * available, component["cutoff"]))


def query_matches(config: dict, query: dict) -> list[tuple[np.ndarray, np.ndarray]]:
    """Return, per component, the direct and derived similarity of each query token
    (a row) to every index token (a column), with the hidden tokens at 0."""
    available = form_token_count > np.bincount(tok_form[query["masked"]], minlength=N_ARCHIVE)
    matches = []
    for component in config["components"]:
        pairs = [component_matches(component, token, available) for token in query["tokens"]]
        direct = np.array([d for d, _ in pairs])[:, tok_form]
        derived = np.array([r for _, r in pairs])[:, tok_form]
        direct[:, query["masked"]], derived[:, query["masked"]] = 0, 0
        matches.append((direct, derived))
    return matches


def person_scores(similarities: np.ndarray, aggregation: str) -> np.ndarray:
    """Turn token similarities (query tokens × index tokens) into one score per person."""
    maxima = np.maximum.reduceat(similarities, seg_start, axis=1)
    surname_run = tok_surname[seg_start]
    given, surname = (np.zeros((len(similarities), len(persons))) for _ in range(2))
    given[:, tok_person[seg_start][~surname_run]] = maxima[:, ~surname_run]
    surname[:, tok_person[seg_start][surname_run]] = maxima[:, surname_run]
    if len(similarities) == 1:
        return np.maximum(given[0], surname[0])
    combine = np.add if aggregation == "sum" else np.multiply
    pairs = combine(given[:, None, :], surname[None, :, :])
    pairs[np.arange(len(given)), np.arange(len(given))] = 0     # i ≠ j
    return pairs.max(axis=(0, 1))


def ranked_persons(scores: np.ndarray) -> np.ndarray:
    """Return the TOP_K best persons with a positive score, ties in person order."""
    found = np.flatnonzero(scores > 0)
    return found[np.lexsort((found, -scores[found]))][:TOP_K]


def query_scores(config: dict, query: dict) -> np.ndarray:
    """Return every person's score for one query under a configuration."""
    matches = query_matches(config, query)
    if config["ranker"] == "K3":
        total = np.zeros(len(persons))
        for direct, derived in matches:
            ranked = ranked_persons(person_scores(np.maximum(direct, config["lambda"] * derived),
                                                  config["aggregation"]))
            total[ranked] += 1.0 / (RRF_K + np.arange(1, len(ranked) + 1))
        return total
    weight = 1.0 if config["ranker"] == "K1" else config["lambda"]
    best = np.max([np.maximum(direct, weight * derived) for direct, derived in matches], axis=0)
    return person_scores(best, config["aggregation"])


def answer_queries(config: dict, query_list: list[dict]) -> list[dict]:
    """Answer every query of `query_list`; return, per query, its ranked persons,
    R@5, reciprocal rank and matching time in milliseconds."""
    answers = []
    for query in query_list:
        started = time.perf_counter()
        ranked = ranked_persons(query_scores(config, query))
        milliseconds = (time.perf_counter() - started) * 1000
        hits = [rank for rank, p in enumerate(ranked, start=1) if p in query["relevant"]]
        answers.append({"query": query, "ranked": ranked, "ms": milliseconds,
                        "r_at_5": sum(r <= 5 for r in hits) / min(5, len(query["relevant"])),
                        "reciprocal_rank": 1 / hits[0] if hits else 0.0})
    return answers


def summarise(answers: list[dict]) -> dict:
    """Return R@5 per query set, and R@5 and MRR macro-averaged over the sets."""
    recall, reciprocal = defaultdict(list), defaultdict(list)
    for answer in answers:
        recall[answer["query"]["set"]].append(answer["r_at_5"])
        reciprocal[answer["query"]["set"]].append(answer["reciprocal_rank"])
    result = {f"r_at_5_{s}": round(float(np.mean(recall[s])), 4) for s in QUERY_SETS}
    result["r_at_5"] = round(float(np.mean([np.mean(recall[s]) for s in QUERY_SETS])), 4)
    result["mrr"] = round(float(np.mean([np.mean(reciprocal[s]) for s in QUERY_SETS])), 4)
    return result


def evaluate(config: dict, query_list: list[dict]) -> dict:
    """Answer every query of `query_list` under `config`; return its scores (see
    `summarise`)."""
    return summarise(answer_queries(config, query_list))


def best_of(tried: list[dict]) -> dict:
    """Return the result with the best macro R@5, ties broken by the MRR."""
    return max(tried, key=lambda r: (r["r_at_5"], r["mrr"]))


print(f"✓ Search engine defined ({len(EMBEDDINGS)} embedding models, {len(LEXICAL)} lexical "
      f"methods, {len(DICTIONARIES)} dictionaries, 3 placements, 3 rankers).")

✓ Search engine defined (4 embedding models, 4 lexical methods, 3 dictionaries, 3 placements, 3 rankers).


**3. Experiment 1**

This cell evaluates each generator alone (ranker K1, sum aggregation) on the validation queries. Each embedding model is tried at every cut-off in `CUTOFFS` and keeps the best one by macro R@5, ties broken by MRR. It carries forward the two best embedding models and the best lexical method, and writes `./outputs/step_5_generator_results.csv`.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment 1  —  every generator alone, its cut-off tuned on validation
# ─────────────────────────────────────────────────────────────────────────────
if "evaluate" not in globals():
    raise RuntimeError("Run the 'Search Engine' cell first.")

# Embedding cut-offs tried. Below about 0.4 the cap of MATCH_CAP matches per
# query token decides which matches are kept, so the grid starts there.
CUTOFFS = (0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90)


def generator_configuration(generator: str, cutoff: float | None) -> dict:
    """Return the configuration of one generator alone."""
    return {"components": [COMPONENT | {"generator": generator, "cutoff": cutoff}],
            "ranker": "K1", "lambda": 1.0, "aggregation": "sum"}


validation_queries = load_queries("validation")
results = []
for _generator in (*EMBEDDINGS, *LEXICAL):
    _kind = "embedding" if _generator in EMBEDDINGS else "lexical"
    results.append(best_of([
        {"generator": _generator, "kind": _kind, "cutoff": _cutoff}
        | evaluate(generator_configuration(_generator, _cutoff), validation_queries)
        for _cutoff in (CUTOFFS if _kind == "embedding" else (None,))]))
results.sort(key=lambda r: (-r["r_at_5"], -r["mrr"]))
carried = ([r["generator"] for r in results if r["kind"] == "embedding"][:2]
           + [r["generator"] for r in results if r["kind"] == "lexical"][:1])
for _result in results:
    _result["carried_forward"] = _result["generator"] in carried
write_csv(OUTPUTS_DIR / "step_5_generator_results.csv", results)

print(f"{RULE}\nExperiment 1: generators on {len(validation_queries)} validation queries "
      f"(R@5)\n{RULE}")
print(f"  {'':<5} {'cut-off':>7}  " + "  ".join(f"{s:>5}" for s in QUERY_SETS) + "   macro    MRR")
for _result in results:
    print(f"  {_result['generator']:<5} {_result['cutoff'] or '':>7}  "
          + "  ".join(f"{_result[f'r_at_5_{s}']:5.3f}" for s in QUERY_SETS)
          + f"   {_result['r_at_5']:.3f}  {_result['mrr']:.3f}"
          + ("   carried" if _result["carried_forward"] else ""))
print(f"{RULE}\n  carried into Experiment 2: {', '.join(carried)}")

────────────────────────────────────────────────────────────────────────
Experiment 1: generators on 513 validation queries (R@5)
────────────────────────────────────────────────────────────────────────
        cut-off    QS1    QS2    QS3    QS4    QS5   macro    MRR
  e1        0.6  0.973  1.000  0.662  0.649  0.855   0.828  0.759   carried
  e3a       0.3  0.982  1.000  0.519  0.649  0.854   0.801  0.726   carried
  e2        0.3  0.973  1.000  0.480  0.558  0.836   0.770  0.723
  l3             0.768  0.989  0.429  0.377  0.623   0.637  0.553   carried
  l4             0.795  0.979  0.377  0.338  0.685   0.634  0.569
  e3b       0.3  0.973  1.000  0.130  0.208  0.756   0.614  0.591
  l2             1.000  1.000  0.000  0.000  0.705   0.541  0.543
  l1             0.804  0.383  0.000  0.000  0.424   0.322  0.283
────────────────────────────────────────────────────────────────────────
  carried into Experiment 2: e1, e3a, l3


**4. Consistency Check**

This cell reads the results back and checks that every generator has one row, that each macro R@5 is the mean of its five query-set values, and that exactly three generators are carried forward.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the results file
# ─────────────────────────────────────────────────────────────────────────────
if "carried" not in globals():
    raise RuntimeError("Run every cell above first.")

_stored = read_csv(OUTPUTS_DIR / "step_5_generator_results.csv")
_problems = [] if sorted(r["generator"] for r in _stored) == sorted((*EMBEDDINGS, *LEXICAL)) \
    else ["generators missing or repeated"]
_problems += [f"{r['generator']}: macro differs from its sets" for r in _stored
              if abs(np.mean([float(r[f"r_at_5_{s}"]) for s in QUERY_SETS]) - float(r["r_at_5"]))
              > 1e-3]
_problems += [] if sum(r["carried_forward"] == "True" for r in _stored) == 3 else [
    "three generators should be carried forward"]
print(f"{RULE}\nConsistency: {OUTPUTS_DIR}/step_5_generator_results.csv\n{RULE}")
for _problem in _problems:
    print(f"  !! {_problem}")
print(f"  {'OK' if not _problems else '!!'} {len(_stored)} generators")
print(f"\n✓ Experiment 1 complete (best: {_stored[0]['generator']}, "
      f"R@5 {float(_stored[0]['r_at_5']):.3f}).")
print(f"  consistent: {not _problems}")

────────────────────────────────────────────────────────────────────────
Consistency: outputs/step_5_generator_results.csv
────────────────────────────────────────────────────────────────────────
  OK 8 generators

✓ Experiment 1 complete (best: e1, R@5 0.828).
  consistent: True
